[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_09/listing_9.3b.ipynb)

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install autoawq transformers

### Listing 9.3 (llm-compressor): Quantizing a merged model with AWQ via LLM Compressor

**Why llm-compressor instead of AutoAWQ?**
AutoAWQ was officially deprecated by its author (Casper Hansen) in 2025. The project
recommended migrating to `llm-compressor`, which absorbed the AWQ algorithm
as its `AWQModifier`. llm-compressor's AWQ produces checkpoints directly
compatible with vLLM through the `compressed-tensors` format.

**AWQ vs GPTQ in llm-compressor**
The two modifiers share the same `oneshot()` API but differ in their strategy:
  - **GPTQ** (Listing 9.2): second-order Hessian-based weight update — accurate,
    but VRAM-intensive during calibration.
  - **AWQ** (this listing): finds per-channel scale factors that reduce activation
    outlier impact *before* rounding, then quantises with a simpler RTN-like step.
    Lower peak VRAM, slightly faster calibration, comparable perplexity.

**Install**
```bash
pip install llmcompressor
```

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.modifiers.quantization import QuantizationModifier

# ── 1. Load the full-precision merged model ───────────────────────────────────
MODEL_DIR = "./merged-model"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    dtype="auto",        # honour the checkpoint's native dtype
    device_map="cuda:0",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# ── 2. Calibration dataset ────────────────────────────────────────────────────
# AWQ calibration passes are lighter than GPTQ's (no Hessian inversion),
# so the same or slightly fewer samples work well.
# 128 samples at 512 tokens is a solid default; use more for domain-specific
# fine-tunes where the original AutoAWQ had no explicit calibration set at all
# (it used a fixed internal dataset).  Using task-relevant data is better.
MAX_SEQ_LENGTH = 512
NUM_CALIBRATION_SAMPLES = 128

raw_texts = [
    "Quantization calibration example text",
    "The quick brown fox jumps over the lazy dog",
    "Large language models are trained on diverse datasets",
]

tokenized = [
    tokenizer(text, truncation=True, max_length=MAX_SEQ_LENGTH)
    for text in raw_texts
]
ds = Dataset.from_list(tokenized)

# ── 3. Configure the AWQ + Quantization recipe ────────────────────────────────
# In llm-compressor, AWQ is implemented as two cooperating modifiers that run
# in sequence during the single oneshot() call:
#
#   AWQModifier  – computes per-channel activation-aware scales and folds them
#                  into the adjacent weight tensors (the "smoothing" step).
#                  It does NOT itself quantize; it prepares the weights so that
#                  a subsequent quantizer makes smaller rounding errors.
#
#                  Key parameters:
#                    mappings  – list of (smooth_layer, balance_layers) pairs
#                                that tell AWQ where to search for salient
#                                channels.  For Mistral/Llama-style models the
#                                built-in registry covers q/k/v/o projections
#                                and MLP gate/up/down automatically.
#                                Leave mappings=None to use the auto-registry.
#
#   QuantizationModifier – performs the actual weight quantization after AWQ
#                          has pre-processed the weights.
#
#   scheme="W4A16_ASYM"  – 4-bit asymmetric (zero-pointed) INT4 weights,
#                          FP16 activations.  This maps to the original
#                          AutoAWQ config: w_bit=4, zero_point=True.
#                          Asymmetric is the historical AWQ default because the
#                          paper found zero_point=True consistently outperforms
#                          symmetric quantization at 4 bits.
#
#                          To replicate symmetric (zero_point=False):
#                            scheme="W4A16"   (symmetric, same as GPTQ default)
#
#   group_size=128 is baked into the W4A16_ASYM built-in scheme (same 128 as
#   the original q_group_size=128).  You can override by building a custom
#   QuantizationArgs if you need 64 or 256.
#
#   version="GEMM" in the original AutoAWQ selected the CUDA kernel backend
#   for inference; llm-compressor is framework-agnostic at save time —
#   vLLM picks the fastest available kernel automatically when loading.

recipe = [
    AWQModifier(
        # mappings=None uses the built-in registry for known architectures
        # (Mistral, Llama, Falcon, Phi, Qwen, …).  Supply explicit mappings
        # for custom or unsupported architectures:
        # mappings=[
        #     AWQMapping(
        #         smooth_layer="re:.*input_layernorm",
        #         balance_layers=["re:.*self_attn.q_proj",
        #                         "re:.*self_attn.k_proj",
        #                         "re:.*self_attn.v_proj"],
        #     ),
        # ],
    ),
    QuantizationModifier(
        targets="Linear",
        scheme="W4A16_ASYM",   # asymmetric 4-bit weights, FP16 activations
        ignore=["lm_head"],    # best practice: skip the output projection
    ),
]

# ── 4. Apply quantization ─────────────────────────────────────────────────────
# oneshot() runs AWQModifier first (scale search + weight folding),
# then QuantizationModifier (actual INT4 rounding).
# The combined effect is equivalent to the original AutoAWQ model.quantize().
#
# Compared with GPTQ (Listing 9.2):
#   - No Hessian computation → lower peak VRAM during calibration.
#   - Slightly faster wall-clock quantization time.
#   - Accuracy is generally comparable; GPTQ can edge out AWQ on
#     perplexity-sensitive tasks.

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQ_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

# ── 5. Save ───────────────────────────────────────────────────────────────────
# The saved checkpoint uses compressed-tensors format:
#   - INT4 weights packed together with per-group zero-points and scales.
#   - A quantization_config block in config.json that vLLM understands natively.
# No separate 'awq' configuration file is needed, unlike the original output.

SAVE_DIR = "./merged-model-awq-llmcompressor"
model.save_pretrained(SAVE_DIR, save_compressed=True)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Quantized model saved to {SAVE_DIR}")
print("Load in vLLM with: LLM(model='./merged-model-awq-llmcompressor')")

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


2026-05-05T21:07:37.817993+0200 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.
2026-05-05T21:07:38.014211+0200 | _make_sampler | WARNING - Requested 128 samples but the provided dataset only has 3 samples.
2026-05-05T21:07:38.014918+0200 | reset | INFO - Compression lifecycle reset
2026-05-05T21:07:38.015857+0200 | from_modifiers | INFO - Creating recipe from modifiers
2026-05-05T21:07:38.017062+0200 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...
2026-05-05T21:07:38.021319+0200 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.0.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-05-05T21:07:38.021684+0200 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.1.self_attn

W0505 21:07:38.086000 587690 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(33/33): Calibrating: 100%|██████████| 3/3 [00:00<00:00, 3950.68it/s]
Smoothing: 0it [00:00, ?it/s]
(33/33): Propagating: 100%|██████████| 3/3 [00:00<00:00, 7507.70it/s]
Smoothing: 0it [00:00, ?it/s]
Updating global scales: 100%|██████████| 225/225 [00:00<00:00, 1063944.08it/s]
Fusing global scales: 647it [00:00, 986088.19it/s]
Calibrating weights: 100%|██████████| 225/225 [00:00<00:00, 1450.52it/s]

2026-05-05T21:08:35.314022+0200 | IndependentPipeline | INFO - Inferred `DataFreePipeline` for `QuantizationModifier`



Calibrating weights: 100%|██████████| 224/224 [00:00<00:00, 1468.05it/s]

2026-05-05T21:08:35.486619+0200 | finalize | INFO - Compression lifecycle finalized for 2 modifiers
2026-05-05T21:08:35.487096+0200 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
2026-05-05T21:08:35.491052+0200 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.



Compressing model: 224it [00:06, 33.47it/s]
/home/lmassaron/code/finetuning/chapter_09/.venv_ch09_llm_compressor/lib/python3.12/site-packages/transformers/modeling_utils.py:3970: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/1 [00:00<?, ?it/s]

Quantized model saved to ./merged-model-awq-llmcompressor
Load in vLLM with: LLM(model='./merged-model-awq-llmcompressor')
